In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from google.colab import files


In [4]:
# --- 1. DATA LOADING ---
print("Upload mnist_train.csv and mnist_test.csv")
uploaded = files.upload()

train_df = pd.read_csv('mnist_train.csv')
test_df = pd.read_csv('mnist_test.csv')

x_train = train_df.drop('label', axis=1).values.reshape(-1, 28, 28, 1) / 255.0
y_train = tf.keras.utils.to_categorical(train_df['label'].values, 10)
x_test = test_df.drop('label', axis=1).values.reshape(-1, 28, 28, 1) / 255.0
y_test = tf.keras.utils.to_categorical(test_df['label'].values, 10)


Upload mnist_train.csv and mnist_test.csv


Saving mnist_test.csv to mnist_test (3).csv
Saving mnist_train.csv to mnist_train (3).csv


In [5]:

# --- 2. ADVANCED AUGMENTATION (To stop 1/7/4/9 confusion) ---
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15, # Crucial for slanted 1s and 7s
    zoom_range=0.15,
    fill_mode='constant',
    cval=0
)


In [6]:

# --- 3. THE "ULTIMATE" ARCHITECTURE ---
model = models.Sequential([
    # First Block: Detecting basic edges
    layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(28, 28, 1)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),

    # Second Block: Detecting complex shapes (loops, intersections)
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    # Third Block: Final Classification
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5), # High dropout to prevent memorizing the dataset
    layers.Dense(10, activation='softmax')
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:

# --- 4. SMART TRAINING ---
# Lowers learning rate when accuracy plateaus to fine-tune the "tricky" numbers
lr_reducer = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=0.00001)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("\nTraining high-accuracy model (approx 5-7 mins)...")
model.fit(datagen.flow(x_train, y_train, batch_size=64),
          epochs=15,
          validation_data=(x_test, y_test),
          callbacks=[lr_reducer])

# --- 5. EXPORT FOR DEPLOYMENT ---
model_name = 'mnist_pro_model.h5'
model.save(model_name)
print(f"\nModel saved as {model_name}")
files.download(model_name) # Automatically downloads to your PC


Training high-accuracy model (approx 5-7 mins)...
Epoch 1/15


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


938/938 ━━━━━━━━━━━━━━━━━━━━ 343s 361ms/step - accuracy: 0.7596 - loss: 0.7935 - val_accuracy: 0.9713 - val_loss: 0.0973 - learning_rate: 0.0010
Epoch 2/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 339s 362ms/step - accuracy: 0.9559 - loss: 0.1415 - val_accuracy: 0.9819 - val_loss: 0.0568 - learning_rate: 0.0010
Epoch 3/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 337s 359ms/step - accuracy: 0.9660 - loss: 0.1104 - val_accuracy: 0.9891 - val_loss: 0.0387 - learning_rate: 0.0010
Epoch 4/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 339s 361ms/step - accuracy: 0.9739 - loss: 0.0858 - val_accuracy: 0.9885 - val_loss: 0.0358 - learning_rate: 0.0010
Epoch 5/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 338s 361ms/step - accuracy: 0.9757 - loss: 0.0804 - val_accuracy: 0.9848 - val_loss: 0.0479 - learning_rate: 0.0010
Epoch 6/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 346s 369ms/step - accuracy: 0.9778 - loss: 0.0735 - val_accuracy: 0.9923 - val_loss: 0.0232 - learning_rate: 0.0010
Epoch 7/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 338s 360ms/step - accuracy: 0.9804 


Model saved as mnist_pro_model.h5


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>